In [3]:
def berlekamp_welch_decoding(u, x_pts, k):
    n = len(x_pts)
    t = (n - k) // 2
    F = u.base_ring()
    R.<X> = PolynomialRing(F)
    
    # 1. Construction du système linéaire M*V = Y
    # Inconnues : Q(X) de degré < k+t (k+t coeffs) et E(X) de degré <= t
    # On fixe E(X) unitaire : E(X) = X^t + e_{t-1}X^{t-1} + ... + e0
    num_vars_Q = k + t
    num_vars_E = t # e0 à e_{t-1}
    
    rows = []
    constants = []
    for i in range(n):
        xi = x_pts[i]
        ui = u[i]
        # Equation: Q(xi) - ui*E(xi) = 0  => Q(xi) - ui*(sum e_j xi^j) = ui * xi^t
        row_Q = [xi^j for j in range(num_vars_Q)]
        row_E = [-ui * xi^j for j in range(num_vars_E)]
        rows.append(row_Q + row_E)
        constants.append(ui * xi^t)
        
    M = matrix(F, rows)
    Y = vector(F, constants)
    print(f"{M} = {Y}")
    # 2. Résolution par Gauss (Temps polynomial)
    sol = M.solve_right(Y)
    
    # 3. Reconstruction des polynômes
    Q = R(list(sol[:num_vars_Q]))
    E = X^t + R(list(sol[num_vars_Q:]))
    
    # 4. Retrouver f = Q / E
    f_recup = Q // E
    return f_recup

In [4]:
# Paramètres
q = 16
F.<a> = GF(q)
n, k = 15, 9
C_RS = codes.ReedSolomonCode(F, n, k)

# Points d'évaluation (support x)
x_pts = C_RS.evaluation_points()

# Envoi d'un message f(X)
R.<X> = PolynomialRing(F)
f = 2*X^2 + a*X + 1
c = vector(F, [f(xi) for xi in x_pts])

# Ajout d'erreurs (jusqu'à t=3)
y = copy(c)
y[0] += 1; y[1] += a; y[2] += a^2

# Décodage via notre fonction Berlekamp-Welch
f_decodes = berlekamp_welch_decoding(y, x_pts, k)
print(f"RS - Message retrouvé : {f_decodes}")

[                1                 1                 1                 1                 1                 1                 1                 1                 1                 1                 1                 1                 a                 a                 a]
[                1                 a               a^2               a^3             a + 1           a^2 + a         a^3 + a^2       a^3 + a + 1           a^2 + 1           a^3 + a       a^2 + a + 1     a^3 + a^2 + a       a^2 + a + 1     a^3 + a^2 + a a^3 + a^2 + a + 1]
[                1               a^2             a + 1         a^3 + a^2           a^2 + 1       a^2 + a + 1 a^3 + a^2 + a + 1           a^3 + 1                 a               a^3           a^2 + a       a^3 + a + 1     a^3 + a^2 + 1                 1               a^2]
[                1               a^3         a^3 + a^2           a^3 + a a^3 + a^2 + a + 1                 1               a^3         a^3 + a^2           a^3 + a a^3 + a^2 + a + 1    

In [5]:
# Paramètres GRS
n_grs, k_grs = 10, 6
F_grs = GF(11)
x_grs = [F_grs(i) for i in range(n_grs)] # Support x
v_weights = [F_grs(1) for _ in range(n_grs)] # Poids v (ici tous à 1)

C_GRS = codes.GeneralizedReedSolomonCode(x_grs, k_grs, v_weights)

# Utilisation du décodeur natif de SageMath (qui peut utiliser BW ou Gao)
y_grs = C_GRS.encode(vector(F_grs, [1, 2, 3, 4, 5, 6]))
y_grs[0] += 1 # 1 erreur

D = C_GRS.decoder("BerlekampWelch")
print(f"GRS - Mot corrigé : {D.decode_to_code(y_grs)}")

GRS - Mot corrigé : (1, 10, 2, 3, 4, 10, 1, 3, 5, 8)


In [9]:
def berlekamp_massey_decoder(u_received, x_support, k):
    n = len(x_support)
    t = (n - k) // 2
    F = u_received.base_ring()
    R.<X> = PolynomialRing(F)
    
    # 1. Construction du polynôme de support Pi(X) [cite: 32]
    Pi = R.prod([X - xi for xi in x_support])
    
    # 2. Construction du polynôme d'interpolation U(X) [cite: 26, 27]
    # Sage utilise l'interpolation de Lagrange en interne
    U = R.lagrange_polynomial(zip(x_support, u_received))
    
    # 3. Algorithme d'Euclide Étendu [cite: 34-45]
    A_prev, A_curr = R(0), R(1)
    B_prev, B_curr = Pi, U
    
    # Condition d'arrêt prématuré : deg(B) < n - t [cite: 53, 54]
    while B_curr.degree() >= n - t:
        Q, r = B_prev.quo_rem(B_curr)
        B_prev, B_curr = B_curr, r
        A_prev, A_curr = A_curr, A_prev - Q * A_curr
        
    f_decoded = B_curr // A_curr
    return f_decoded

# --- TEST ---
F.<a> = GF(16)
n, k = 15, 9
# Correction ici : conversion directe en liste
x_support = list(F)[:n] 

R.<X> = PolynomialRing(F)
f_orig = X^2 + a*X + 1
c = vector(F, [f_orig(xi) for xi in x_support])

# Ajout de 3 erreurs (t = (15-9)//2)
u = copy(c)
u[1], u[3], u[7] = u[1]+1, u[3]+a, u[7]+a^2

f_res = berlekamp_massey_decoder(u, x_support, k)
print(f"Message retrouvé : {f_res}")

Message retrouvé : X^2 + a*X + 1
